In [1]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
import accelerate
import torch
import pandas as pd
import numpy as np
os.environ["HUGGINGFACE_TOKEN"] = "INSERT TOKEN"
cache_dir=' '
#model_name = "microsoft/phi-4"
#model_name="google/gemma-2-27b-it"
#model_name="meta-llama/Llama-3.2-3B-Instruct"
#model_name="meta-llama/Llama-3.1-70B-Instruct"
#model_name="CohereLabs/aya-expanse-32b"
#model_name="tiiuae/falcon-7b-instruct"
model_name="mistralai/Mixtral-8x7B-Instruct-v0.1"
tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)
model = AutoModelForCausalLM.from_pretrained(
    model_name, torch_dtype=torch.bfloat16, device_map="auto", use_auth_token=os.getenv("HUGGINGFACE_TOKEN"),cache_dir=cache_dir)


#model_name = "tiiuae/Falcon3-Mamba-7B-Instruct"

#model = AutoModelForCausalLM.from_pretrained(
#    model_name,
#    torch_dtype=torch.bfloat16,
#    device_map="auto",
# cache_dir=cache_dir
#)
#tokenizer = AutoTokenizer.from_pretrained(model_name)


/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/tokenization_auto.py:823: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/home/rag24/.conda/envs/readability/lib/python3.11/site-packages/transformers/models/auto/auto_factory.py:471: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(


Loading checkpoint shards:   0%|          | 0/19 [00:00<?, ?it/s]

In [2]:
import pandas as pd
import numpy as np
file_path="readme_en_resultsv2.csv"
df=pd.read_csv(file_path)
df=df.drop(["Unnamed: 0"],axis=1)
#df=df.iloc[:,:7]
df.head()

,Domain,Sub-domain,Paragraph,Context,Sentence,Rating,LLAMA_70B_vanilla,LLAMA_70B_avg,LLAMA_70B_entropy,LLAMA_3B_vanilla,...,noscale_gemma_9b_entropy,noscale_gemma_2b_vanilla,noscale_gemma_2b_avg,noscale_gemma_2b_entropy,noscale_gemma_27b_vanilla,noscale_gemma_27b_avg,noscale_gemma_27b_entropy,noscale_llama_70b_vanilla,noscale_llama_70b_avg,noscale_llama_70b_entropy
0,Textbooks,Agriculture,This pricing rule relates the price markup ove...,NaN,This pricing rule relates the price markup ove...,4,5,4.500000,0.058932,5,...,0.052784,6,5.901578,0.061167,6,5.814177,0.046698,7,6.924142,0.022831
1,Textbooks,Agriculture,The cross price elasticity of supply captures ...,NaN,The cross price elasticity of supply captures ...,4,4,4.448104,0.058473,5,...,0.055986,4,4.334472,0.115152,6,5.964300,0.041959,7,6.602686,0.057126
2,Textbooks,Agriculture,Input prices (Pi) are important determinants o...,Prices of related goods (Pr) represent prices ...,Complements in production are goods that are p...,4,3,2.602686,0.057126,3,...,0.052528,5,4.647744,0.093244,4,3.671221,0.072524,4,3.647809,0.063990
3,Textbooks,Agriculture,"If the price ceiling is set at P’, then the eq...",NaN,"If the price ceiling is set at P’, then the eq...",5,4,4.260829,0.048796,3,...,0.058820,6,5.223063,0.094478,6,5.705768,0.072289,7,6.551895,0.058473
4,Textbooks,Agriculture,A large share of citrus fruit in the US is gro...,NaN,A large share of citrus fruit in the US is gro...,2,2,1.500000,0.058932,3,...,0.020512,2,1.663865,0.068400,2,2.002220,0.003787,2,2.000000,-0.000000


In [27]:
df.columns

Index(['Domain', 'Sub-domain', 'Paragraph', 'Context', 'Sentence', 'Rating',
       'LLAMA_70B_vanilla', 'LLAMA_70B_avg', 'LLAMA_70B_entropy',
       'LLAMA_3B_vanilla', 'LLAMA_3B_avg', 'LLAMA_3B_entropy',
       'LLAMA_8B_vanilla', 'LLAMA_8B_avg', 'LLAMA_8B_entropy',
       'Mixtral_vanilla', 'Mixtral_avg', 'Mixtral_entropy', 'Aya_8b_vanilla',
       'Aya_8b_avg', 'Aya_8b_entropy', 'Aya_32b_vanilla', 'Aya_32b_avg',
       'Aya_32b_entropy', 'Falcon_vanilla', 'Falcon_avg', 'Falcon_entropy',
       'Gemma_27b_vanilla', 'Gemma_27b_avg', 'Gemma_27b_entropy',
       'Gemma_9b_vanilla', 'Gemma_9b_avg', 'Gemma_9b_entropy',
       'Gemma_2b_vanilla', 'Gemma_2b_avg', 'Gemma_2b_entropy', 'phi_vanilla',
       'phi_avg', 'phi_entropy', 'noscale_mixtral_vanilla',
       'noscale_mixtral_avg', 'noscale_mixtral_entropy',
       'noscale_aya_32B_vanilla', 'noscale_aya_32B_avg',
       'noscale_aya_32B_entropy', 'noscale_aya_8B_vanilla',
       'noscale_aya_8B_avg', 'noscale_aya_8B_entropy',
       '

In [3]:
## Gets readability scores without eliciting model confidence

## MODELS: LLAMA 3B or 8B or 70B or Mixtral or Aya 8B or 32B or phi 4 or gemma 2b/9b/27B
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,4]
    score=df.iloc[i,5]
        ## CEFR PROMPT
    #prompt = f"Rate the readability of the text between 1 (very easy) and 6 (very challenging) using the following scale: 1 = Can understand very short, simple texts a single phrase at a time, picking up familiar names, words and basic phrases and rereading as required. 2 = Can understand short, simple texts on familiar matters of a concrete type 3 = Can read straightforward factual texts on subjects related to his/her field and interest with a satisfactory level of comprehension. 4 = Can read with a large degree of independence, adapting style and speed of reading to different texts and purpose 5 = Can understand in detail lengthy, complex texts, whether or not they relate to his/her own area of speciality, provided he/she can reread difficult sections. 6 = Can understand and interpret critically virtually all forms of the written language including abstract, structurally complex, or highly colloquial literary and non-literary writings. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"
       ## NO SCALE PROMPT
    prompt=f"Rate the readability of each sentence with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=4,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-1][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM')
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens,top6_probs.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                break
        scores_llm_avg.append(avg)
    real_scores.append(score)
    entropies.append(normalized_entropy)
        #try:
        #    count+=int(tokenizer.decode(outputs[0][-1]))
        #    count1+=1
        #except ValueError:
        #    try:
            #print(tokenizer.decode(outputs[0][-4:]),int(tokenizer.decode(outputs[0][-1])))
        #        count+=int(tokenizer.decode(outputs[0][-2]))
         #       count1+=1
         #   except ValueError:
         #       print('error')
         #       print(tokenizer.decode(outputs[0][-5:]))
   # try:
   #     scores_llm.append(count/count1)
   #     scores.append(score)
   # except ValueError:
   #     print('Big error')
   # except ZeroDivisionError:
   #     print('Zero error')
   #     scores_llm.append('na')
   #     scores.append('na')

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
df['noscale_mixtral_vanilla']=scores_llm_vanilla
df['noscale_mixtral_avg']=scores_llm_avg
df['noscale_mixtral_entropy']=entropies
df.to_csv("readme_en_resultsv2.csv")

From v4.47 onwards, when a model cache is to be returned, `generate` will return a `Cache` instance instead by default (as opposed to the legacy tuple of tuples format). If you want to keep returning the legacy format, please set `return_legacy_cache=True`.


ERROR - FIRST TOKEN NOT NUM
0.5715060787158112 295 0.5818457055051095 295


In [32]:
## Gets readability scores and confidence scores
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
verbal_confs=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,4]
    score=df.iloc[i,5]
        ## CEFR PROMPT
    prompt = f"Rate the readability of the text between 1 (very easy) and 6 (very challenging) using the following scale: 1 = Can understand very short, simple texts a single phrase at a time, picking up familiar names, words and basic phrases and rereading as required. 2 = Can understand short, simple texts on familiar matters of a concrete type 3 = Can read straightforward factual texts on subjects related to his/her field and interest with a satisfactory level of comprehension. 4 = Can read with a large degree of independence, adapting style and speed of reading to different texts and purpose 5 = Can understand in detail lengthy, complex texts, whether or not they relate to his/her own area of speciality, provided he/she can reread difficult sections. 6 = Can understand and interpret critically virtually all forms of the written language including abstract, structurally complex, or highly colloquial literary and non-literary writings. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Answer with this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    
    ## NO SCALE PROMPT
    
    #prompt=f"Rate the readability of each sentence with a whole number value between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Additionally, state how confident you are that your rating will align with human raters, with a whole number value between 1 and 9. Answer with this format: Answer: [SCORE] Confidence: [Confidence Score] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    input_ids = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt",return_dict=True).to("cuda")
    #inputs = tokenizer(prompt, return_tensors="pt").to("cuda")  # Move to GPU if available
    #print(i)
    #for j in range(5):
    outputs = model.generate(**input_ids,max_new_tokens=10,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    logits_last=scores[-3][0]
    logits_fifth_to_last = scores[-7][0]
    import torch
    probs_last = torch.softmax(logits_last, dim=-1)
    probs_fifth_to_last = torch.softmax(logits_fifth_to_last, dim=-1)
    entropy = -torch.sum(probs_fifth_to_last * torch.log(probs_fifth_to_last + 1e-12)).item()
    vocab_size = probs_fifth_to_last.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
    top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())

    top6_probs_fifth_to_last, top6_ids_fifth_to_last = torch.topk(probs_fifth_to_last, 6)
    top6_tokens_fifth_to_last=tokenizer.convert_ids_to_tokens(top6_ids_fifth_to_last.tolist())

    
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens_fifth_to_last[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM')
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens_fifth_to_last,top6_probs_fifth_to_last.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
        
    avg2=0
    flag2=False
    try:
        vanilla_conf=int(top6_tokens_last[0])
    except ValueError:
        try:
            logits_last=scores[-2][0]
            probs_last = torch.softmax(logits_last, dim=-1)
            top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
            top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
            vanilla_conf=int(top6_tokens_last[0])
        except ValueError:
            try:
                logits_last=scores[-1][0]
                probs_last = torch.softmax(logits_last, dim=-1)
                top6_probs_last, top6_ids_last = torch.topk(probs_last, 6)
                top6_tokens_last=tokenizer.convert_ids_to_tokens(top6_ids_last.tolist())
                vanilla_conf=int(top6_tokens_last[0])
            except ValueError:
                
                print('ERROR - FIRST CONFIDENCE TOKEN NOT NUM',tokenizer.decode(outputs['sequences'][0][-10:]))
                flag2=True
                verbal_confs.append('na')
    if flag2==False:
        for (n,p) in zip(top6_tokens_last,top6_probs_last.tolist()):
            try:
                avg2+=int(n)*p
            except ValueError:
                #print(avg)
                break
        verbal_confs.append(avg2)
    real_scores.append(score)
    entropies.append(normalized_entropy)

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
filtered_entropies, filtered_verbal_confs = zip(*[(e,v) for e,v in zip(entropies, verbal_confs) if e != 'na' and v != 'na'])
print(np.corrcoef(filtered_entropies, filtered_verbal_confs)[0, 1])

df['conf_aya_32b_vanilla']=scores_llm_vanilla
df['conf_aya_32b_avg']=scores_llm_avg
df['conf_aya_32b_entropy']=entropies
df['conf_aya_32b_verbal_conf']=verbal_confs
#print(corrs)
df.to_csv("readme_hi_resultsv2.csv")

ERROR - FIRST CONFIDENCE TOKEN NOT NUM Answer: 2-3
Confidence: 
0.6582553243895335 163 0.7163903446766811 163
-0.11722350018081892


In [29]:
df

,Domain,Sub-domain,Paragraph,Context,Sentence,Rating,Transliteration,AYA_8b_vanilla,AYA_8b_avg,AYA_8b_entropy,...,conf_llama_8b_entropy,conf_llama_8b_verbal_conf,conf_llama_3b_vanilla,conf_llama_3b_avg,conf_llama_3b_entropy,conf_llama_3b_verbal_conf,conf_noscale_llama_3b_vanilla,conf_noscale_llama_3b_avg,conf_noscale_llama_3b_entropy,conf_noscale_llama_3b_verbal_conf
0,Wikipedia,Arts and Culture,जब इमारत के सामने और बगल के हिस्से को बनाया जा...,NaN,जब इमारत के सामने और बगल के हिस्से को बनाया जा...,4,False,3,3.000426,0.000308,...,0.074436,8.000000,5,4.551897,0.058473,8.000000,7,6.964532,0.074580,7.448105
1,Wikipedia,Arts and Culture,कच्चे भोज्य पदार्थों को अग्नि के संयोग से पकाक...,कच्चे भोज्य पदार्थों को अग्नि के संयोग से पकाक...,सीधे अग्नि के संयोग से भी पकाया जाता है.,2,False,3,2.957919,0.014024,...,0.054975,8.000000,5,4.606825,0.076566,8.000000,7,6.730348,0.102092,7.602686
2,Wikipedia,Arts and Culture,"एसिड-मुक्त, अभिलेखीय गुणवत्ता वाला कागज़, लकड़...",NaN,"एसिड-मुक्त, अभिलेखीय गुणवत्ता वाला कागज़, लकड़...",5,True,3,3.000180,0.000139,...,0.073503,8.000000,4,4.151356,0.076651,7.867036,6,6.045725,0.084090,8.0
3,Wikipedia,Arts and Culture,खौलते पानी में भोज्य पदार्थ को पकाने की क्रिया...,प्राय: सभी भोज्य पदार्थ इस प्रकार पकाए जा सकते...,सब प्रकार की साग सब्जियाँ उबालकर खाने से अधिक ...,2,False,3,2.651347,0.051935,...,0.029590,8.158869,4,4.302940,0.052146,8.000000,6,6.226165,0.075865,7.205371
4,Wikipedia,Geography,जलवायु को प्रभावित करनेवाले बाह्य कारक  किसी ...,NaN,जलवायु को प्रभावित करनेवाले बाह्य कारक  किसी ...,5,False,3,2.999651,0.000299,...,0.057126,8.000000,5,4.500000,0.058932,8.000000,7,7.448104,0.058473,6.847735
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
158,Social Media,Twitter,लाॅकडाउन में साईकिल से 1500 किलोमीटर चल कर अपन...,NaN,लाॅकडाउन में साईकिल से 1500 किलोमीटर चल कर अपन...,6,True,4,4.453004,0.072368,...,0.052146,8.188723,6,5.651355,0.054975,8.500000,8,7.7773,0.045087,7.051101
159,Social Media,Twitter,आंदोलन के 17वें दिन किसानों ने कई टोल प्लाजा क...,NaN,आंदोलन के 17वें दिन किसानों ने कई टोल प्लाजा क...,4,True,3,2.994781,0.002622,...,0.048796,8.000000,4,4.302941,0.052146,7.867036,6,5.687664,0.099132,7.602686
160,Social Media,Twitter,"लखनऊ में सपा कार्यकर्ताओं पर लाठीचार्ज, 30 विध...",NaN,"लखनऊ में सपा कार्यकर्ताओं पर लाठीचार्ज, 30 विध...",4,True,3,2.999426,0.000698,...,0.083323,8.000000,5,4.651355,0.054975,8.000000,7,7.260829,0.048796,7.30271
161,Social Media,Twitter,कोटा आर्मी इलाके से पकडा गया संदिग्ध युवक इमरा...,NaN,कोटा आर्मी इलाके से पकडा गया संदिग्ध युवक इमरा...,5,True,3,2.999425,0.000466,...,0.081796,8.000000,5,4.697059,0.052146,8.000000,7,7.158869,0.037221,6.896209


In [13]:
##FALCON -- failed model
corrs=[]   
scores_llm_avg=[]
scores_llm_vanilla=[]
real_scores=[]
entropies=[]
import pandas as pd
import numpy as np
for i in range(len(df)):
    count=0
    count1=0
    s=df.iloc[i,4]
    score=df.iloc[i,5]

    prompt = f"Rate the readability of the text between 1 (very easy) and 6 (very challenging) using the following scale: 1 = Can understand very short, simple texts a single phrase at a time, picking up familiar names, words and basic phrases and rereading as required. 2 = Can understand short, simple texts on familiar matters of a concrete type 3 = Can read straightforward factual texts on subjects related to his/her field and interest with a satisfactory level of comprehension. 4 = Can read with a large degree of independence, adapting style and speed of reading to different texts and purpose 5 = Can understand in detail lengthy, complex texts, whether or not they relate to his/her own area of speciality, provided he/she can reread difficult sections. 6 = Can understand and interpret critically virtually all forms of the written language including abstract, structurally complex, or highly colloquial literary and non-literary writings. You may use both the provided scale and your own understanding of readability to determine the most appropriate score. Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    prompt=f" Rate the readability of each passage between 1 (very easy to understand) and 9 (very difficult to understand). Consider factors such as sentence structure, vocabulary or grammar complexity, and overall clarity, as well as your own understanding of readability.  Where on the scale of readability does this text rate: '{s}'. Answer with this format: Answer: [SCORE] Explanation: [EXPLANATION]"
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(messages,tokenize=False,add_generation_prompt=True)
    model_inputs = tokenizer([text], return_tensors="pt").to("cuda")

    outputs = model.generate(**model_inputs, max_new_tokens=4,return_dict_in_generate=True,output_scores=True,pad_token_id=tokenizer.eos_token_id)
    scores = outputs.scores  
    #second to last token from generating 5 will have number, so change to 4 and get last num
    logits = scores[-1][0] 
    import torch
    probs = torch.softmax(logits, dim=-1)
    entropy = -torch.sum(probs * torch.log(probs + 1e-12)).item()
    vocab_size = probs.shape[0]
    normalized_entropy = entropy / torch.log(torch.tensor(vocab_size, dtype=torch.float)).item()
    #six possible number values so we take top6...it will not always consider all 6. Some have 0.0 probability
    top6_probs, top6_ids = torch.topk(probs, 6)
    top6_tokens=tokenizer.convert_ids_to_tokens(top6_ids.tolist())
    avg=0
    flag=False
    try:
        vanilla=int(top6_tokens[0])
        scores_llm_vanilla.append(vanilla)
    except ValueError:
        print('ERROR - FIRST TOKEN NOT NUM',top6_tokens)
        scores_llm_vanilla.append('na')
        scores_llm_avg.append('na')
        flag=True
    if flag==False:
        for (n,p) in zip(top6_tokens,top6_probs.tolist()):
            try:
                avg+=int(n)*p
            except ValueError:
                #print(avg)
                break
        scores_llm_avg.append(avg)
    real_scores.append(score)
    entropies.append(normalized_entropy)

filtered_scores, filtered_scores_vanilla,filtered_scores_avg = zip(*[(s, llm_v,llm_a) for s, llm_v,llm_a in zip(real_scores, scores_llm_vanilla,scores_llm_avg) if s != 'na' and llm_v != 'na'])
print(np.corrcoef(filtered_scores, filtered_scores_vanilla)[0, 1],len(filtered_scores_vanilla),np.corrcoef(filtered_scores, filtered_scores_avg)[0, 1],len(filtered_scores_avg))
#corrs.append(np.corrcoef(filtered_scores, filtered_scores_llm)[0, 1])
#print(np.corrcoef(scores, scores_llm)[0, 1])
df['Falcon_vanilla']=scores_llm_vanilla
df['Falcon_avg']=scores_llm_avg
df['Falcon_entropy']=entropies
#print(corrs)
df.to_csv("readme_fr_resultsv2.csv")

ERROR - FIRST TOKEN NOT NUM ['Ġof', 'Ġwritten', 'Ġvery', 'Ġhighly', 'Ġat', 'Ġquite']
ERROR - FIRST TOKEN NOT NUM ['Ġof', 'Ġwritten', 'Ġat', 'Ġhighly', 'Ġvery', 'Ġabout']
ERROR - FIRST TOKEN NOT NUM ['Ġwritten', 'Ġof', 'Ġat', 'Ġvery', 'Ġhighly', 'Ġrated']
ERROR - FIRST TOKEN NOT NUM ['Ġwritten', 'Ġat', 'Ġhighly', 'Ġvery', 'Ġrated', 'Ġeasy']
ERROR - FIRST TOKEN NOT NUM ['Ġwritten', 'Ġof', 'Ġat', 'Ġhighly', 'Ġvery', 'Ġrated']
ERROR - FIRST TOKEN NOT NUM ['opause', 'Ġmenopause', 'op', 'ostasis', 'ographie', 'oscope']
ERROR - FIRST TOKEN NOT NUM ['Ġwritten', 'Ġhighly', 'Ġof', 'Ġvery', 'Ġat', 'Ġnot']
ERROR - FIRST TOKEN NOT NUM ['Ġwritten', 'Ġabout', 'Ġhighly', 'Ġvery', 'Ġat', 'Ġnot']
ERROR - FIRST TOKEN NOT NUM ['Ġwritten', 'Ġat', 'Ġeasy', 'Ġof', 'Ġvery', 'Ġhighly']
ERROR - FIRST TOKEN NOT NUM ['Ġwritten', 'Ġvery', 'Ġof', 'Ġhighly', 'Ġeasy', 'Ġat']
ERROR - FIRST TOKEN NOT NUM ['Ġwritten', 'Ġat', 'Ġof', 'Ġvery', 'Ġhighly', 'Ġdifficult']
ERROR - FIRST TOKEN NOT NUM ['RE', 'OR', 'RES', 'UR', '


KeyboardInterrupt

